# 🔎 02. Analiza słownikowa
## 🔎 Metody mieszane w analizie tekstu: od słowników do BERT

**Cel:** zmienić jawne reguły dopasowania słów w przejrzysty wskaźnik. Kategorie poniżej są przykładami dydaktycznymi, **nie zwalidowanymi skalami psychologicznymi**.

**Pomiar:** pojedyncze słowo → kategoria badacza → wynik dokumentu → opis grup → ręczna ocena trafności. Wskaźnik odzwierciedla obecność wybranych oznak leksykalnych, nie sam konstrukt psychologiczny. W tym module nie wykonujemy testów hipotez między grupami.

# 🧭 Jak wykonać ten notebook

1. Otwórz plik `.ipynb` w Google Colab i zapisz własną kopię na Dysku Google.
2. **Wystarczy CPU.** Komórka to jeden blok tekstu albo kodu. Kod uruchamiasz przyciskiem ▶ po lewej lub Shift+Enter.
3. Uruchom pierwszą komórkę kodu. Jeżeli zainstaluje biblioteki i pokaże 🔄, uruchom ponownie sesję z menu **Środowisko wykonawcze**. Potem zacznij od pierwszej komórki. Restart zachowuje pliki, ale usuwa zmienne z pamięci.
4. Wykonuj kod **od góry, bez pomijania komórek**. Dane pobiorą się automatycznie z [repozytorium prowadzącego](https://github.com/bartlomiejnowak-ux/PSPS-2026). Domyślnie niczego nie wgrywasz. W razie awarii pobierania komórka podaje instrukcję ręcznego wgrania.
5. Poczekaj, aż obracający się znacznik przy komórce zniknie. Pierwsze pobranie modelu i obliczenia mogą potrwać kilka minut, a BERTopic dłużej. Nie klikaj wielokrotnie ▶.
6. ✅ oznacza sukces, 🔎 wskazuje co przeczytać, ⚠️ ważne ograniczenie, ✏️ ćwiczenie. Emoji nie zmieniają działania kodu.
7. Na końcu pobierz ZIP wyników. Sam zapis notebooka na Dysku nie zachowuje plików z tymczasowej sesji.


### 🛠️ Gdy coś nie działa

| Objaw | Co zrobić |
|---|---|
| `NameError` lub „nie zdefiniowano” | Pominięto wcześniejszy krok albo zrestartowano sesję. Wykonaj kod od początku. |
| `ModuleNotFoundError` / błąd wersji biblioteki | Uruchom instalację, zrestartuj sesję i wykonaj komórki od góry. |
| Brak pliku / zła kolumna | Ponów komórkę pobierania danych. Awaryjnie wybierz `cleaned_topic_modeling_dataset(1).csv` z repozytorium. |
| Błąd pobierania modelu | Sprawdź połączenie, zaczekaj i ponów komórkę pobierania. Nie zmieniaj nazwy modelu. |
| Błąd po zmianie parametru | Cofnij zmianę lub przywróć wartości pokazane w komentarzach i wykonaj dalsze komórki kolejno. |
| Sesja wygasła | Połącz ponownie, uruchom notebook od początku i ponownie wykonaj komórkę pobierania danych. |

Nie przechodź dalej po czerwonym błędzie. Czytaj ostatnią linijkę komunikatu. W tej kopii wyniki pojawią się dopiero po uruchomieniu kodu. Liczby na slajdach pochodzą ze sprawdzonego wcześniejszego wykonania; przy innych ustawieniach lub środowisku wynik może się różnić.

## 📖 Słowa i skróty używane w tym module

- **Konstrukt:** pojęcie teoretyczne, np. wsparcie społeczne.
- **Trafność:** czy wskaźnik mierzy to, co chcemy nim mierzyć.
- **Leksykalny:** dotyczący słów.
- **Normalizacja względem długości:** przeliczenie trafień na stałą liczbę słów.
- **Bootstrap:** wielokrotne losowanie dokumentów ze zwracaniem, aby oszacować niepewność.
- **95% CI:** 95-procentowy przedział ufności: przy wielokrotnym powtarzaniu poprawnej procedury około 95% takich przedziałów obejmie prawdziwy parametr.
- **Pokrycie:** jaką część poszukiwanych zjawisk wychwytuje lista słów.
- **NaN:** brak zdefiniowanego wyniku, a nie zero.
- **Fleksja:** odmiana słowa, np. worry / worries.

# 🔎 1. Przygotowanie
Uruchom komórkę instalacji poniżej. W Colab wszystko wykonujesz w przeglądarce.

### ▶️ Krok kodu 1

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** ✅ Biblioteki gotowe albo instrukcja jednorazowego restartu 🔄.

In [ ]:
import sys, subprocess, importlib.util, importlib.metadata as metadata
from pathlib import Path
IN_COLAB = importlib.util.find_spec('google.colab') is not None if importlib.util.find_spec('google') else False
PACKAGES = ['numpy==2.5.3', 'pandas==3.0.5', 'matplotlib==3.11.2', 'spacy==3.8.16', 'https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl']
if sys.version_info < (3, 12):
    raise RuntimeError('⛔ Ten zestaw wersji wymaga Python 3.12 lub nowszego. Wybierz zgodne środowisko.')
def installed(spec):
    name, expected = ('en-core-web-sm', '3.8.0') if spec.startswith('https:') else spec.split('==')
    try: return metadata.version(name) == expected
    except metadata.PackageNotFoundError: return False
needed = [spec for spec in PACKAGES if not installed(spec)]
if needed and IN_COLAB:
    print('⏳ Instalacja bibliotek. Poczekaj na zakończenie tej komórki.')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *PACKAGES])
    raise RuntimeError('🔄 Instalacja zakończona. Uruchom ponownie sesję z menu Środowisko wykonawcze, a następnie wykonaj komórki od początku. To jednorazowy krok po instalacji.')
if needed:
    raise RuntimeError('⛔ Brakuje wymaganych wersji. Lokalnie użyj pliku requirements właściwego modułu. W Colab komórka instaluje je sama. Braki: ' + ', '.join(needed))
print('✅ Biblioteki gotowe. Możesz uruchomić następną komórkę.')


### ▶️ Krok kodu 2

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** komunikat lub wyniki tekstowe pod komórką.

In [ ]:
MODULE = "02_DICTIONARY_ANALYSIS"
from pathlib import Path
import json, re, time
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
START = time.perf_counter()
HERE = Path.cwd()
ROOT = HERE.parent if HERE.name.startswith(("01_", "02_", "03_", "04_")) else HERE
(ROOT / "data").mkdir(exist_ok=True)
OUT = ROOT / MODULE / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
TARGET = {0:"Stress", 1:"Depression", 2:"Bipolar", 3:"Personality Disorder", 4:"Anxiety"}
RNG = np.random.default_rng(42)
plt.rcParams.update({"figure.dpi":120, "axes.spines.top":False, "axes.spines.right":False})
print("Folder wyników:", OUT)

# 🔎 2. Wczytanie danych
Korzystamy z tego samego korpusu. Tokeny w mianowniku zawierają również stopwords.

### ▶️ Krok kodu 3

Uruchom raz i poczekaj. **Oczekiwany efekt:** automatyczne pobranie danych i komunikat ✅. Przy awarii ustaw w kodzie `DATA_SOURCE = 'upload'` i wybierz **`cleaned_topic_modeling_dataset(1).csv`** z repozytorium prowadzącego.

In [ ]:
# 📥 Dane z repozytorium prowadzącego. Domyślnie niczego nie wgrywasz ręcznie.
import io, urllib.request, hashlib
DATA_SOURCE = 'github'  # awaryjnie zmień na 'upload' i uruchom komórkę ponownie
GITHUB_COMMIT = '1cb710169713a4f8ea5a0d839be6d6e1df8ab078'
GITHUB_URL = f'https://raw.githubusercontent.com/bartlomiejnowak-ux/PSPS-2026/{GITHUB_COMMIT}/cleaned_topic_modeling_dataset%281%29.csv'
EXPECTED_SHA256 = '4fb7bdc1779127e875ce3ea7fd571f85885b700937b9439bd606b7c2d67d7972'
INPUT_NAME = 'cleaned_topic_modeling_dataset(1).csv'
if IN_COLAB:
    if DATA_SOURCE == 'github':
        print('⏳ Pobieranie danych z GitHub…')
        try:
            with urllib.request.urlopen(GITHUB_URL, timeout=60) as response:
                data_bytes = response.read()
        except Exception as exc:
            raise RuntimeError('⛔ Nie udało się pobrać danych. Sprawdź internet i ponów tę komórkę. Awaryjnie ustaw DATA_SOURCE = "upload" i wybierz plik z repozytorium prowadzącego.') from exc
        if hashlib.sha256(data_bytes).hexdigest() != EXPECTED_SHA256:
            raise ValueError('⛔ Pobrany plik nie zgadza się ze sprawdzoną wersją. Nie kontynuuj analizy na tym pliku.')
    elif DATA_SOURCE == 'upload':
        from google.colab import files
        print('📂 Wybierz cleaned_topic_modeling_dataset(1).csv z repozytorium prowadzącego.')
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise ValueError('⛔ Wybierz dokładnie jeden plik i ponów tę komórkę.')
        data_bytes = next(iter(uploaded.values()))
    else:
        raise ValueError('⛔ DATA_SOURCE musi mieć wartość "github" albo "upload".')
    try:
        source_df = pd.read_csv(io.BytesIO(data_bytes), keep_default_na=False)
    except Exception as exc:
        raise ValueError('⛔ Nie można odczytać pliku. Wgraj cleaned_topic_modeling_dataset(1).csv z repozytorium prowadzącego.') from exc
    missing_columns = {'target', 'document'} - set(source_df.columns)
    if source_df.empty or missing_columns:
        raise ValueError(f'⛔ Pusty lub niewłaściwy plik. Brakujące kolumny: {sorted(missing_columns)}')
    data_folder = ROOT / 'data'
    data_folder.mkdir(parents=True, exist_ok=True)
    (data_folder / INPUT_NAME).write_bytes(data_bytes)
    print(f'✅ Dane gotowe: {len(source_df)} wierszy. Źródło: {DATA_SOURCE}.')

# Moduły 02–03 odtwarzają przygotowanie z modułu 01 w osobnej sesji Colab.
if IN_COLAB:
    import re, json, spacy
    print('⏳ Przygotowanie tokenów i form podstawowych. Poczekaj kilka minut.')
    source_df.insert(0, 'source_row', np.arange(len(source_df)))
    def clean_for_workshop(text):
        text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
        text = re.sub(r'(?<!\w)(?:u/|@)[A-Za-z0-9_-]+', ' ', text)
        return re.sub(r'\s+', ' ', text).strip()
    source_df['document_clean'] = source_df.document.map(clean_for_workshop)
    source_df['tokens'] = source_df.document_clean.map(lambda t: re.findall(r"[A-Za-z]+(?:'[A-Za-z]+)?", t.lower()))
    source_df['n_words'] = source_df.tokens.map(len)
    prep_nlp = spacy.load('en_core_web_sm', disable=['parser','ner'])
    all_lemmas, content_lemmas = [], []
    for doc in prep_nlp.pipe(source_df.document_clean.tolist(), batch_size=64):
        all_lemmas.append([t.lemma_.lower() for t in doc if t.is_alpha])
        content_lemmas.append(' '.join(t.lemma_.lower() for t in doc if t.is_alpha and not t.is_stop))
    source_df['tokens_lemma'] = all_lemmas
    source_df['document_lemma'] = content_lemmas
    for column in ['tokens','tokens_lemma']:
        source_df[column] = source_df[column].map(lambda values: json.dumps(values, ensure_ascii=False))
    source_df.to_csv(data_folder / 'workshop_preprocessed.csv', index=False)
    print('✅ Przygotowano dane do tego modułu. Nie trzeba wcześniej uruchamiać modułu 01.')


### ▶️ Krok kodu 4

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** tabela z wynikami lub przykładami.

In [ ]:
input_path = ROOT / "data/workshop_preprocessed.csv"
if not input_path.exists():
    raise FileNotFoundError("Najpierw wykonaj notebook 01: zapisuje data/workshop_preprocessed.csv.")
df = pd.read_csv(input_path, keep_default_na=False)
df["tokens"] = df.tokens.map(json.loads)
df["tokens_lemma"] = df.tokens_lemma.map(json.loads)
assert df.target.isin(TARGET).all() and df.source_row.is_unique
assert np.array_equal(df.n_words, df.tokens.map(len))
df["group"] = df.target.map(TARGET)
display(df[["source_row", "target", "n_words"]].head())

# 🔎 3. Kontrola danych
Różnice długości mogą prowadzić do różnic liczby trafień nawet przy takim samym udziale słów kategorii.

### ▶️ Krok kodu 5

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** tabela z wynikami lub przykładami.

In [ ]:
display(df.groupby("group").n_words.agg(["size", "mean", "median", "std"]).reindex(TARGET.values()).round(2))
print("Duplikaty pozostają:", df.document.duplicated().sum())

# 🔎 4. Analiza
## 🔎 4.1 Małe, przejrzyste słowniki
Każda kategoria to zbiór haseł. Jedno słowo może należeć do kilku kategorii; sumy kategorii nie muszą być rozłączne.

### ▶️ Krok kodu 6

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** komunikat lub wyniki tekstowe pod komórką.

In [ ]:
DICTIONARIES = {
    "ANXIETY": ["anxious", "anxiety", "panic", "fear", "worried", "worry", "nervous", "scared"],
    "SADNESS": ["sad", "sadness", "depressed", "hopeless", "cry", "lonely", "miserable", "empty"],
    "SOCIAL_SUPPORT": ["friend", "family", "support", "help", "talk", "together", "relationship"]}
display(pd.Series(DICTIONARIES))
print("Dokładne dopasowanie:", [(t, t in DICTIONARIES["ANXIETY"]) for t in ["worry", "worries", "worried"]])

## 🔎 4.2 Liczba i częstość względna
Liczymy każde wystąpienie, a nie liczbę różnych haseł. `rate = hits / total words × 100`. Dla zera słów wskaźnik jest niezdefiniowany, więc pozostaje NaN.

### ▶️ Krok kodu 7

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** tabela z wynikami lub przykładami.

In [ ]:
for category, words in DICTIONARIES.items():
    terms = set(words)
    df[f"{category}_count"] = df.tokens.map(lambda tokens: sum(t in terms for t in tokens))
    df[f"{category}_rate"] = 100 * df[f"{category}_count"] / df.n_words.replace(0, np.nan)
display(df[["n_words", "ANXIETY_count", "ANXIETY_rate"]].head(8))
print("Pomijane w średnich wskaźników, bo brak tokenów:", df.n_words.eq(0).sum())
normalization_demo = pd.DataFrame({"hits":[2, 2], "words":[20, 200]})
normalization_demo["rate"] = 100 * normalization_demo.hits / normalization_demo.words
display(normalization_demo)

## 🔎 4.3 Surowa częstość słowa a wynik kategorii
Częstość „panic” pyta o jedno słowo. Kategoria ANXIETY agreguje z góry zdefiniowane formy. Żaden z tych wskaźników sam nie mierzy rozpoznania klinicznego.

### ▶️ Krok kodu 8

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** tabela z wynikami lub przykładami.

In [ ]:
df["panic_count"] = df.tokens.map(lambda tokens: tokens.count("panic"))
display(df[["source_row", "panic_count", "ANXIETY_count", "ANXIETY_rate"]].sort_values("ANXIETY_rate", ascending=False).head(6))

## 🔎 4.4 Fleksja i wspólne przetwarzanie haseł
Lematyzacja może zwiększyć pokrycie, ale zmienia definicję dopasowania. Poniżej wynik pomocniczy: także hasła słownika lematyzujemy. Mianownikiem jest teraz liczba lematyzowanych tokenów, więc różnica obejmuje tokenizację.

### ▶️ Krok kodu 9

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** tabela z wynikami lub przykładami.

In [ ]:
import spacy
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
lemma_terms = {t.lemma_.lower() for word in DICTIONARIES["ANXIETY"] for t in nlp(word) if t.is_alpha}
df["ANXIETY_lemma_count"] = df.tokens_lemma.map(lambda tokens: sum(t in lemma_terms for t in tokens))
df["ANXIETY_lemma_rate"] = 100 * df.ANXIETY_lemma_count / df.tokens_lemma.map(len).replace(0, np.nan)
display(df[["ANXIETY_count", "ANXIETY_lemma_count"]].describe())

# 🔎 5. Wykresy
## 🔎 5.1 Opis i niepewność
Bootstrap dokumentów: 1000 prób, 95% przedziały percentylowe. To ilustracja niepewności przy niezależnych dokumentach, nie losowa próba populacji Reddita. Powtórzone teksty i nieznani powtarzający się autorzy ograniczają tę interpretację.

### ▶️ Krok kodu 10

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** przygotowanie funkcji lub ustawień do dalszych kroków; brak wydruku jest prawidłowy.

In [ ]:
def mean_ci(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    means = [RNG.choice(values, len(values), replace=True).mean() for _ in range(1000)]
    return pd.Series({"n":len(values), "mean":values.mean(), "sd":values.std(ddof=1),
                      "low":np.quantile(means, .025), "high":np.quantile(means, .975)})

## 🔎 Średnie, SD i przedziały
Te same kategorie target i porządek grup w całym warsztacie.

### ▶️ Krok kodu 11

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** tabela z wynikami lub przykładami.

In [ ]:
rows = []
for category in DICTIONARIES:
    for target, group in df.groupby("target"):
        rows.append({"category":category, "target":target, "group":TARGET[target], **mean_ci(group[f"{category}_rate"])})
summary = pd.DataFrame(rows)
display(summary.round(3))
summary.to_csv(OUT / "dictionary_by_target.csv", index=False)

## 🔎 Średnie z 95% CI
Przedziały dla pojedynczych średnich nie są testem wszystkich różnic między grupami.

### ▶️ Krok kodu 12

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** wykres.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharey=True)
for ax, category in zip(axes, DICTIONARIES):
    tab = summary[summary.category.eq(category)]
    ax.errorbar(tab["mean"], range(5), xerr=[tab["mean"]-tab.low, tab.high-tab["mean"]], fmt="o", color="#2B8C8E", capsize=3)
    ax.set_yticks(range(5), TARGET.values()); ax.set_title(category); ax.set_xlabel("Trafienia na 100 słów")
fig.tight_layout(); fig.savefig(OUT / "dictionary_means_ci.png"); plt.show()

## 🔎 Rozkłady
Średnia może ukrywać dużo zer i kilka wysokich wartości.

### ▶️ Krok kodu 13

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** wykres.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.boxplot([df.loc[df.target.eq(i), "ANXIETY_rate"].dropna() for i in TARGET], tick_labels=list(TARGET.values()), showfliers=False)
ax.set(ylabel="ANXIETY: trafienia na 100 słów", title="Rozkład wskaźnika (punkty odstające ukryte)")
fig.tight_layout(); fig.savefig(OUT / "dictionary_distribution.png"); plt.show()
display(df.groupby("target").ANXIETY_count.apply(lambda x: x.eq(0).mean()).rename("udział_zer"))

## 🔎 5.2 Wysokie wyniki wymagają czytania
Przykłady pochodzą z korpusu i są skrócone. Krótki tekst może mieć wysoki wskaźnik przy jednym trafieniu.

### ▶️ Krok kodu 14

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** tabela z wynikami lub przykładami.

In [ ]:
examples = df.nlargest(4, "ANXIETY_rate")[["source_row", "group", "n_words", "ANXIETY_rate", "document"]].copy()
examples["document"] = examples.document.str.slice(0, 350)
display(examples)
examples.to_json(OUT / "high_examples.json", orient="records", force_ascii=False)

# 🔎 6. Interpretacja
## 🔎 Negacja, wieloznaczność, fałszywe trafienia i pominięcia
Poniższe **skonstruowane przykłady** mają znaną intencję. Trafienie w „not anxious” nie potwierdza lęku; „empty bottle” nie oznacza smutku. W korpusie trafienia wymagają ręcznej oceny, a nie automatycznego uznania za błędy.

### ▶️ Krok kodu 15

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** tabela z wynikami lub przykładami.

In [ ]:
failure_cases = ["I am not anxious.", "The bottle is empty.", "Please help me delete this file.", "My heart races before every meeting."]
failure_table = pd.DataFrame({"text":failure_cases})
for category, words in DICTIONARIES.items():
    failure_table[category] = failure_table.text.map(lambda t: sum(w in words for w in re.findall(r"[a-z]+", t.lower())))
display(failure_table)
context_candidates = df[df.document.str.contains(r"not.{0,25}(?:anxious|sad)|empty|help", case=False, regex=True)].head(3)
display(context_candidates[["source_row", "document"]].assign(document=lambda x:x.document.str.slice(0, 350)))

## 🔎 Trafność słownika i zależność od dziedziny
Trafność wymaga zdefiniowania konstruktu, ręcznego kodowania próbki i sprawdzenia fałszywych trafień/pominięć. Wyższy wskaźnik może oznaczać inną konwencję językową, nie silniejszą emocję.

**Opcjonalnie:** [LIWC](https://www.liwc.app/dictionaries) wymaga właściwej licencji; [NRC Emotion Lexicon](https://saifmohammad.com/WebPages/NRC-Emotion-Lexicon.htm) opisuje skojarzenia słów z emocjami; [Moral Foundations Dictionary](https://moralfoundations.org/other-materials/) dotyczy języka moralnego. Sprawdź wersję, warunki użycia, język i domenę. Nie dołączamy komercyjnych słowników.

## 🔎 Zapis
Kolumny kategorii są osobnymi wskaźnikami, a nie wynikiem jednej skali.

### ▶️ Krok kodu 16

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** komunikat lub wyniki tekstowe pod komórką.

In [ ]:
score_columns = [c for c in df if c.endswith(("_count", "_rate"))]
df[["source_row", "target", "n_words"]+score_columns].to_csv(OUT / "dictionary_scores.csv", index=False)
failure_table.to_json(OUT / "failure_cases.json", orient="records", force_ascii=False)
(OUT / "summary.json").write_text(json.dumps({"documents":len(df), "runtime_seconds":time.perf_counter()-START,
    "zero_word_documents":int(df.n_words.eq(0).sum())}), encoding="utf-8")

# 🔎 7. Ćwiczenie
1. Przeczytaj pięć trafień „help”: które dotyczą wsparcia społecznego?
2. Dodaj jedno uzasadnione hasło i zapisz regułę przed obejrzeniem różnic grup.
3. Czy „not anxious” powinno dawać zero? Jak obsłużysz „not only anxious”?
4. Porównaj wynik po usunięciu duplikatów; nazwij zmienioną jednostkę analizy.

**Przejście:** sentyment dodaje reguły dotyczące tonu wypowiedzi. Nadal wymaga oceny w kontekście.

## 💾 Pobranie wyników

Uruchom tę komórkę po ukończeniu analizy. Utworzy ZIP i w Colab rozpocznie pobieranie. Jeśli przeglądarka je zablokuje, odszukaj ZIP w panelu Pliki i pobierz ręcznie. Zachowaj też własną kopię notebooka.

In [ ]:
import shutil
bundle = Path('wyniki_modul_02')
bundle.mkdir(exist_ok=True)
if not Path(OUT).exists():
    raise RuntimeError('⛔ Najpierw wykonaj komórki analizy i zapisu wyników.')
shutil.copytree(OUT, bundle / 'tabele_i_wykresy', dirs_exist_ok=True)
archive = shutil.make_archive(str(bundle), 'zip', bundle)
print('✅ Plik wyników:', archive)
if IN_COLAB:
    from google.colab import files
    files.download(archive)
